# Sparkle V2 — Fase 3.2b: Fusión apariencia + geometría (el experimento que decide el proyecto)

**Entorno de ejecución:** Google Colab (GPU T4) o Kaggle Notebooks (GPU)
**Requiere:** embeddings ya extraídos por `Sparkle_V2_Fase3_2a_Extraccion_Embeddings_CNN.ipynb` (`embeddings_cnn/{Train,Validation,Test}/*.npz`) + las features geométricas de `features_v2/` (mismas de Fase 3.0/3.1).

**Objetivo (Roadmap Fase 3.2, semanas 3-4 del plan):** entrenar el modelo de fusión (§3.3) y correr la ablación **embeddings-solo vs geométricas-solo vs fusión** — esta es la evidencia central del diagnóstico del plan (§0): la apariencia es la palanca principal, la geometría por sí sola tiene un techo ~50%.

**Criterio de salida:** ≥ 68–72% accuracy binaria en Test oficial con brecha > +12 pts vs. trivial. Si se cumple, es el resultado que sostiene la tesis (según el plan, "el experimento que decide el proyecto").

In [ ]:
!pip install -q scikit-learn tensorflow

import os, glob, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_DIR  = '/content/drive/MyDrive/SparkleV2'
FEATURES_DIR = f'{PROJECT_DIR}/features_v2'
EMB_DIR      = f'{PROJECT_DIR}/embeddings_cnn'
CKPT_DIR     = f'{PROJECT_DIR}/checkpoints_fase3_2'
os.makedirs(CKPT_DIR, exist_ok=True)

SEED = 42
N_FRAMES = 60
N_EMB_FRAMES = 16
np.random.seed(SEED); tf.random.set_seed(SEED)

## 1. Carga y alineación de las dos fuentes (embeddings + geométricas)

Cada clip tiene un `.npz` de geométricas (`features_v2`, 22 features crudas) y otro de embeddings (`embeddings_cnn`, 16×128). Se cargan ambas por `ClipID` y se descartan los clips que falten en cualquiera de las dos fuentes (deben ser pocos: mismos videos, mismo detector facial subyacente).

In [ ]:
DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen',
                  'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']

def load_aligned_split(split):
    geom_files = {os.path.splitext(os.path.basename(f))[0]: f
                  for f in glob.glob(f'{FEATURES_DIR}/{split}/*.npz')}
    emb_files  = {os.path.splitext(os.path.basename(f))[0]: f
                  for f in glob.glob(f'{EMB_DIR}/{split}/*.npz')}
    common_ids = sorted(set(geom_files) & set(emb_files))
    print(f'{split}: {len(geom_files)} geometricas, {len(emb_files)} embeddings, '
          f'{len(common_ids)} en comun')

    X_geom, X_emb, y3, eng, conf, frus, ids = [], [], [], [], [], [], []
    feat_names = None
    for cid in common_ids:
        dg = np.load(geom_files[cid], allow_pickle=True)
        de = np.load(emb_files[cid], allow_pickle=True)
        seq = dg['sequence'].astype(np.float32)
        emb = de['embeddings'].astype(np.float32)
        if seq.shape != (N_FRAMES, 22) or emb.shape != (N_EMB_FRAMES, 128):
            continue
        if feat_names is None:
            feat_names = list(dg['feature_names'])
        X_geom.append(seq); X_emb.append(emb)
        b = int(dg['boredom'])
        y3.append(0 if b == 0 else (1 if b == 1 else 2))
        eng.append(int(dg['engagement'])); conf.append(int(dg['confusion'])); frus.append(int(dg['frustration']))
        ids.append(cid)
    return (np.stack(X_geom), np.stack(X_emb), np.array(y3, dtype=np.int64),
            np.array(eng), np.array(conf), np.array(frus), np.array(ids), feat_names)

Xg_tr, Xe_tr, y3_tr, eng_tr, conf_tr, frus_tr, id_tr, FEATS22 = load_aligned_split('Train')
Xg_va, Xe_va, y3_va, eng_va, conf_va, frus_va, id_va, _        = load_aligned_split('Validation')
Xg_te, Xe_te, y3_te, eng_te, conf_te, frus_te, id_te, _        = load_aligned_split('Test')

In [ ]:
# Re-split por participante (Train+Validation), Test oficial intacto -- igual criterio que Fase 3.0/3.1
Xg_pool = np.concatenate([Xg_tr, Xg_va]); Xe_pool = np.concatenate([Xe_tr, Xe_va])
y3_pool = np.concatenate([y3_tr, y3_va])
eng_pool = np.concatenate([eng_tr, eng_va]); conf_pool = np.concatenate([conf_tr, conf_va]); frus_pool = np.concatenate([frus_tr, frus_va])
id_pool = np.concatenate([id_tr, id_va])
participant_pool = np.array([cid[:6] for cid in id_pool])
y_bin_pool = (y3_pool > 0).astype(np.int64)

def interpolate_sequence(seq):
    df = pd.DataFrame(seq)
    df = df.interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean_geom(X, train_medians=None, nan_frame_thresh=0.30):
    keep, X_clean = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > nan_frame_thresh:
            continue
        seq = interpolate_sequence(X[i])
        if np.isnan(seq).any() and train_medians is not None:
            idx = np.where(np.isnan(seq))
            seq[idx] = np.take(train_medians, idx[1])
        X_clean.append(seq); keep.append(i)
    return np.stack(X_clean), np.array(keep)

def clean_emb(X, train_medians=None, nan_frame_thresh=0.5):
    """Frames sin rostro detectado -> NaN (ver Fase 3.2a). Interpolar igual que geometricas;
    umbral mas laxo (50%) porque el muestreo es disperso (16 frames) y perder 1-2 pesa mas."""
    keep, X_clean = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > nan_frame_thresh:
            continue
        seq = interpolate_sequence(X[i])
        if np.isnan(seq).any() and train_medians is not None:
            idx = np.where(np.isnan(seq))
            seq[idx] = np.take(train_medians, idx[1])
        X_clean.append(seq); keep.append(i)
    return np.stack(X_clean), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X, idx):
    d_ = np.zeros_like(X[:, :, idx])
    d_[:, 1:, :] = np.diff(X[:, :, idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx0, va_idx0 = next(sgkf.split(Xg_pool, y_bin_pool, groups=participant_pool))
print(f'Pool: {len(id_pool)} clips | train_split={len(tr_idx0)} val_split={len(va_idx0)}')

In [ ]:
# Limpieza + deltas + escalado (geometricas) -- fit SOLO en el split de train
geom_medians = np.nanmedian(Xg_pool[tr_idx0].reshape(-1, 22), axis=0)
Xg_tr_c, keep_tr_g = clean_geom(Xg_pool[tr_idx0], geom_medians)
Xg_va_c, keep_va_g = clean_geom(Xg_pool[va_idx0], geom_medians)
Xg_tr_d = add_deltas(Xg_tr_c, delta_idx); Xg_va_d = add_deltas(Xg_va_c, delta_idx)
geom_scaler = StandardScaler().fit(Xg_tr_d.reshape(-1, 32))
def scale_geom(X):
    N, T, F = X.shape
    return geom_scaler.transform(X.reshape(-1, F)).reshape(N, T, F).astype(np.float32)
Xg_tr_s, Xg_va_s = scale_geom(Xg_tr_d), scale_geom(Xg_va_d)

# Limpieza + escalado (embeddings) -- fit SOLO en el split de train
emb_medians = np.nanmedian(Xe_pool[tr_idx0].reshape(-1, 128), axis=0)
Xe_tr_c, keep_tr_e = clean_emb(Xe_pool[tr_idx0], emb_medians)
Xe_va_c, keep_va_e = clean_emb(Xe_pool[va_idx0], emb_medians)
emb_scaler = StandardScaler().fit(Xe_tr_c.reshape(-1, 128))
def scale_emb(X):
    N, T, F = X.shape
    return emb_scaler.transform(X.reshape(-1, F)).reshape(N, T, F).astype(np.float32)
Xe_tr_s, Xe_va_s = scale_emb(Xe_tr_c), scale_emb(Xe_va_c)

# Interseccion de indices validos entre ambas fuentes (clean_geom y clean_emb pueden descartar clips distintos)
keep_tr = np.intersect1d(keep_tr_g, keep_tr_e)
keep_va = np.intersect1d(keep_va_g, keep_va_e)
def reindex(X_full, keep_full, keep_final):
    pos = {k: i for i, k in enumerate(keep_full)}
    idx = [pos[k] for k in keep_final]
    return X_full[idx]
Xg_tr_s = reindex(Xg_tr_s, keep_tr_g, keep_tr); Xe_tr_s = reindex(Xe_tr_s, keep_tr_e, keep_tr)
Xg_va_s = reindex(Xg_va_s, keep_va_g, keep_va); Xe_va_s = reindex(Xe_va_s, keep_va_e, keep_va)
y_tr = y_bin_pool[tr_idx0][keep_tr]; y_va = y_bin_pool[va_idx0][keep_va]
print(f'Train final: {len(y_tr)} | Val final: {len(y_va)}')

# Test oficial: mismo tratamiento con estadisticos del train re-split
Xg_te_c, keep_te_g = clean_geom(Xg_te, geom_medians)
Xg_te_d = add_deltas(Xg_te_c, delta_idx); Xg_te_s = scale_geom(Xg_te_d)
Xe_te_c, keep_te_e = clean_emb(Xe_te, emb_medians)
Xe_te_s = scale_emb(Xe_te_c)
keep_te = np.intersect1d(keep_te_g, keep_te_e)
Xg_te_s = reindex(Xg_te_s, keep_te_g, keep_te); Xe_te_s = reindex(Xe_te_s, keep_te_e, keep_te)
y_bin_te_full = (y3_te > 0).astype(np.int64)
y_te = y_bin_te_full[keep_te]

trivial_test = max(Counter(y_te.tolist()).values()) / len(y_te)
print(f'Test oficial: {len(y_te)} clips | trivial={trivial_test:.4f}')

## 2. Arquitectura de fusión (§3.3 del plan)

Dos ramas: embeddings CNN (16×128, upsample a 60) + geométricas (60×32, `BatchNorm`) → concat por frame → TCN (igual que Fase 3.1) → attention pooling → cabezas binaria + ordinal (misma solución `Dense(K-1)` de Fase 3.1 por compatibilidad TFLite, ver ese notebook para el detalle del hallazgo).

In [ ]:
def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout=0.3, l2=1e-4):
    reg = keras.regularizers.l2(l2)
    prev = x
    h = layers.Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation_rate, kernel_regularizer=reg)(x)
    h = layers.BatchNormalization()(h); h = layers.Activation('relu')(h); h = layers.Dropout(dropout)(h)
    h = layers.Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation_rate, kernel_regularizer=reg)(h)
    h = layers.BatchNormalization()(h); h = layers.Activation('relu')(h); h = layers.Dropout(dropout)(h)
    if prev.shape[-1] != filters:
        prev = layers.Conv1D(filters, 1, padding='same')(prev)
    return layers.Activation('relu')(layers.Add()([prev, h]))

class AttentionPooling(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs); self.score_dense = layers.Dense(1)
    def call(self, x):
        scores = self.score_dense(x); weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(x * weights, axis=1), tf.squeeze(weights, axis=-1)

class CoralHead(layers.Layer):
    def __init__(self, num_classes, **kwargs):
        super().__init__(**kwargs); self.num_classes = num_classes
    def build(self, input_shape):
        self.dense = layers.Dense(self.num_classes - 1, name='coral_thresholds')
    def call(self, x):
        return tf.nn.sigmoid(self.dense(x))

def ordinal_loss(y_true_3level, p_hat, n_classes=3):
    y_true_3level = tf.cast(y_true_3level, tf.float32)
    losses = [keras.losses.binary_crossentropy(tf.cast(y_true_3level > k, tf.float32), p_hat[:, k])
              for k in range(n_classes - 1)]
    return tf.reduce_mean(tf.add_n(losses) / (n_classes - 1))

def build_fusion_model(input_mode, n_geom_feats=32, n_emb_frames=16, emb_dim=128,
                        n_frames=60, filters=64, dilations=(1, 2, 4, 8), dropout=0.3, l2=1e-4):
    """input_mode: 'fusion' | 'geom_only' | 'emb_only' -- para la ablacion de la seccion 4."""
    inputs = []
    branches = []
    if input_mode in ('fusion', 'emb_only'):
        inp_emb = keras.Input(shape=(n_emb_frames, emb_dim), name='embeddings_cnn')
        inputs.append(inp_emb)
        emb_up = layers.UpSampling1D(size=n_frames // n_emb_frames)(inp_emb)
        cur_len = emb_up.shape[1]
        if cur_len < n_frames:
            emb_up = layers.ZeroPadding1D((0, n_frames - cur_len))(emb_up)
        elif cur_len > n_frames:
            emb_up = layers.Cropping1D((0, cur_len - n_frames))(emb_up)
        branches.append(emb_up)
    if input_mode in ('fusion', 'geom_only'):
        inp_geom = keras.Input(shape=(n_frames, n_geom_feats), name='geometricas')
        inputs.append(inp_geom)
        branches.append(layers.BatchNormalization()(inp_geom))

    x = branches[0] if len(branches) == 1 else layers.Concatenate(axis=-1)(branches)
    for d in dilations:
        x = tcn_residual_block(x, filters, 3, d, dropout=dropout, l2=l2)
    pooled, attn_w = AttentionPooling(name='attention_pooling')(x)
    h = layers.Dense(32, activation='relu', kernel_regularizer=keras.regularizers.l2(l2))(pooled)
    h = layers.Dropout(dropout)(h)
    binary_out = layers.Dense(1, activation='sigmoid', name='binary')(h)
    coral_out = CoralHead(3, name='coral')(h)
    return keras.Model(inputs, [binary_out, coral_out])

## 3. Entrenamiento y ablación — embeddings-solo vs. geométricas-solo vs. fusión

Esta tabla es la evidencia central del plan: si `fusion` supera claramente a `geom_only` (que ya se midió en Fase 3.0/3.1 en el orden de 55-63%), confirma que la apariencia es la palanca que faltaba.

In [ ]:
def smoothed_class_weights(y, beta=0.5):
    counts = Counter(y.tolist()); total = len(y)
    w = {int(c): (total / counts[c]) ** beta for c in counts}
    mean_w = np.mean(list(w.values()))
    return {c: w[c] / mean_w for c in w}

def get_inputs(mode, Xe, Xg):
    if mode == 'fusion':
        return [Xe, Xg]
    elif mode == 'emb_only':
        return [Xe]
    elif mode == 'geom_only':
        return [Xg]

def train_and_eval_fusion(mode, seed, epochs=120, patience=15, batch_size=64):
    tf.random.set_seed(seed); np.random.seed(seed)
    model = build_fusion_model(mode)
    optimizer = keras.optimizers.Adam(1e-3, clipnorm=1.0)
    cw = smoothed_class_weights(y_tr, beta=0.5)
    sw_tr = np.array([cw[int(v)] for v in y_tr], dtype='float32')

    Xtr_in = get_inputs(mode, Xe_tr_s, Xg_tr_s)
    Xva_in = get_inputs(mode, Xe_va_s, Xg_va_s)
    y3_tr_pool = y3_pool[tr_idx0][keep_tr]   # para la perdida ordinal

    ds = tf.data.Dataset.from_tensor_slices((tuple(Xtr_in), y_tr.astype('float32'),
                                              y3_tr_pool.astype('int64'), sw_tr))
    ds = ds.shuffle(4096, seed=SEED).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    best_f1, wait, best_w = -1.0, 0, None
    for epoch in range(epochs):
        for Xb, yb, y3b, swb in ds:
            Xb_list = list(Xb) if isinstance(Xb, tuple) else [Xb]
            with tf.GradientTape() as tape:
                p_bin, p_coral = model(Xb_list if len(Xb_list) > 1 else Xb_list[0], training=True)
                l_bin = tf.reduce_mean(swb * keras.losses.binary_crossentropy(yb[:, None], p_bin))
                l_ord = ordinal_loss(y3b, p_coral)
                total = l_bin + 0.5 * l_ord
            grads = tape.gradient(total, model.trainable_variables)
            grads, _ = tf.clip_by_global_norm(grads, 1.0)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
        Xva_input = Xva_in if len(Xva_in) > 1 else Xva_in[0]
        p_val = (model.predict(Xva_input, verbose=0)[0].ravel() > 0.5).astype(int)
        f1_val = f1_score(y_va, p_val, average='macro')
        if f1_val > best_f1 + 1e-4:
            best_f1, wait, best_w = f1_val, 0, model.get_weights()
        else:
            wait += 1
        if wait >= patience:
            break
    model.set_weights(best_w)
    return model, best_f1, epoch + 1

SEEDS = [0, 1, 2, 3, 4]
ablation_results = []
models_by_mode = {}
for mode in ['geom_only', 'emb_only', 'fusion']:
    mode_results = []
    for seed in SEEDS:
        model, best_val_f1, stopped_epoch = train_and_eval_fusion(mode, seed)
        Xte_in = get_inputs(mode, Xe_te_s, Xg_te_s)
        Xte_input = Xte_in if len(Xte_in) > 1 else Xte_in[0]
        p_te = (model.predict(Xte_input, verbose=0)[0].ravel() > 0.5).astype(int)
        acc = accuracy_score(y_te, p_te)
        r = {'mode': mode, 'seed': seed, 'acc': acc,
             'f1_macro': f1_score(y_te, p_te, average='macro'),
             'balanced_acc': balanced_accuracy_score(y_te, p_te),
             'gap_vs_trivial': acc - trivial_test, 'stopped_epoch': stopped_epoch}
        mode_results.append(r)
        ablation_results.append(r)
        print(f"[{mode}] seed={seed}: acc={acc:.4f} gap={r['gap_vs_trivial']:+.4f}")
        if seed == SEEDS[0]:
            models_by_mode[mode] = model
    df_mode = pd.DataFrame(mode_results)
    print(f"=== {mode}: acc={df_mode['acc'].mean():.4f}+/-{df_mode['acc'].std():.4f} "
          f"gap={df_mode['gap_vs_trivial'].mean():+.4f}+/-{df_mode['gap_vs_trivial'].std():.4f} ===\n")

df_ablation = pd.DataFrame(ablation_results)
df_ablation.to_csv(f'{CKPT_DIR}/fase3_2_ablacion_5semillas.csv', index=False)
print(df_ablation.groupby('mode')[['acc', 'f1_macro', 'balanced_acc', 'gap_vs_trivial']].agg(['mean', 'std']))

## 4. Export a TFLite del modelo de fusión (2 modelos: extractor CNN + fusión temporal)

El extractor CNN (`Sparkle_V2_Fase3_2a`) ya se exporta por separado. Acá se exporta el modelo de fusión temporal (batch fijo=1, misma mitigación que en Fase 3.0/3.1 para evitar operadores dinámicos no soportados).

In [ ]:
best_model_fusion = models_by_mode['fusion']

def build_fusion_export(batch_size=1):
    inp_emb = keras.Input(shape=(N_EMB_FRAMES, 128), batch_size=batch_size, name='embeddings_cnn')
    inp_geom = keras.Input(shape=(N_FRAMES, 32), batch_size=batch_size, name='geometricas')
    emb_up = layers.UpSampling1D(size=N_FRAMES // N_EMB_FRAMES)(inp_emb)
    cur_len = emb_up.shape[1]
    if cur_len < N_FRAMES:
        emb_up = layers.ZeroPadding1D((0, N_FRAMES - cur_len))(emb_up)
    geom_bn = layers.BatchNormalization()(inp_geom)
    x = layers.Concatenate(axis=-1)([emb_up, geom_bn])
    for d in (1, 2, 4, 8):
        x = tcn_residual_block(x, 64, 3, d, dropout=0.0, l2=0.0)
    pooled, attn_w = AttentionPooling(name='attention_pooling')(x)
    h = layers.Dense(32, activation='relu')(pooled)
    binary_out = layers.Dense(1, activation='sigmoid', name='binary')(h)
    coral_out = CoralHead(3, name='coral')(h)
    return keras.Model([inp_emb, inp_geom], [binary_out, coral_out, attn_w])

export_model = build_fusion_export(batch_size=1)
src_layers = {l.name: l for l in best_model_fusion.layers}
for layer in export_model.layers:
    if layer.name in src_layers and src_layers[layer.name].get_weights():
        try:
            layer.set_weights(src_layers[layer.name].get_weights())
        except ValueError:
            print(f'  (omitido, shape distinta): {layer.name}')

converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_float = converter.convert()
print(f'TFLite fusion float: {len(tflite_float)/1024:.1f} KB')

rep_sample_emb = Xe_tr_s[np.random.default_rng(SEED).choice(len(Xe_tr_s), size=min(200, len(Xe_tr_s)), replace=False)]
rep_sample_geom = Xg_tr_s[np.random.default_rng(SEED).choice(len(Xg_tr_s), size=min(200, len(Xg_tr_s)), replace=False)]
def representative_dataset():
    for i in range(len(rep_sample_emb)):
        yield [rep_sample_emb[i:i+1].astype('float32'), rep_sample_geom[i:i+1].astype('float32')]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
tflite_int8 = converter_int8.convert()
print(f'TFLite fusion int8: {len(tflite_int8)/1024:.1f} KB')
with open(f'{CKPT_DIR}/fusion_model.tflite', 'wb') as f:
    f.write(tflite_int8)

## 5. Criterios de salida de Fase 3.2

| Métrica | Umbral del plan | Resultado |
| :--- | :--- | :--- |
| Accuracy binaria Test oficial, modo `fusion` (media 5 semillas) | ≥ 68-72% | ver tabla de la sección 3 |
| Brecha vs. trivial, modo `fusion` | > +12 pts | ver tabla de la sección 3 |
| `fusion` > `geom_only` de forma clara | — | evidencia de que la apariencia es la palanca principal (diagnóstico §0 del plan) |
| TFLite (extractor + fusión) | tamaño total < ~3-4 MB (backbone ~2MB + fusión <1MB) | ver sección 4 y el notebook 3.2a |

Si se cumple: pasar a **Fase 3.3** (cuantización int8 con QAT si hace falta, benchmark de latencia/energía en hardware real, evaluación final con 5 semillas + bootstrap CI + análisis por participante — ver Roadmap del plan). Este es, según el plan, el resultado que sostiene la tesis: *"mismo orden de exactitud que enfoques full-frame, sin que un solo píxel salga del dispositivo"* (§7).